# 🧠 MODMA 3-Channel EEG — VMD+SampEn+LightGBM Seed-Stability Test

## 0 · Setup

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
print("✅ GPU:", r.stdout.strip() if r.returncode==0 else "Not found — CPU mode")

In [ ]:
%%capture
!pip install vmdpy xgboost lightgbm scikit-learn numpy scipy pandas matplotlib seaborn tqdm
print("✅ Done")

In [ ]:
import os, json, warnings, time
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import welch
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, recall_score, roc_curve)
import lightgbm as lgb
from vmdpy import VMD
from tqdm import tqdm

plt.rcParams.update({'figure.dpi':150,'font.size':11,'font.family':'serif',
                     'axes.spines.top':False,'axes.spines.right':False})

np.random.seed(42)
print("✅ All imports OK")

## 1 · Mount Drive & Load Arrays

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

EEG_DIR    = '/content/drive/MyDrive/Depression_FL_Project/data/processed/eeg'
OUTPUT_DIR = '/content/drive/MyDrive/Depression_FL_Project/data/processed/eeg_vmd_stability'
os.makedirs(OUTPUT_DIR, exist_ok=True)

FS = 250
CHANNELS = ['Fp1', 'Fpz', 'Fp2']

def load_arrays(base):
    d = {}
    for sp in ['train','val','test']:
        for k in ['X','y','ids']:
            fp = os.path.join(base, f'{k}_{sp}.npy')
            if os.path.exists(fp):
                d[f'{k}_{sp}'] = np.load(fp, allow_pickle=True)
    return d

d = load_arrays(EEG_DIR)
X_train,y_train,ids_train = d['X_train'],d['y_train'],d['ids_train']
X_val,  y_val,  ids_val   = d['X_val'],  d['y_val'],  d['ids_val']
X_test, y_test, ids_test  = d['X_test'], d['y_test'],  d['ids_test']

X_all   = np.concatenate([X_train,   X_val,   X_test],  axis=0)
y_all   = np.concatenate([y_train,   y_val,   y_test],  axis=0)
ids_all = np.concatenate([ids_train, ids_val, ids_test], axis=0)

unique_sids = np.unique(ids_all)
y_subj = np.array([int(y_all[ids_all==s][0]) for s in unique_sids])

print(f"✅ Loaded: {len(X_all)} epochs, {len(unique_sids)} subjects "
      f"(MDD={np.sum(y_subj==1)}, HC={np.sum(y_subj==0)})")

## 2 · Evaluation Helper

In [ ]:
def evaluate(name, y_true, y_pred, y_prob=None, verbose=True):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    sens = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    spec = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    auc  = (roc_auc_score(y_true, y_prob)
            if y_prob is not None and len(np.unique(y_true))>1
            else float('nan'))
    cm   = confusion_matrix(y_true, y_pred)
    if verbose:
        print(f"\n{'─'*50}\n  {name}")
        print(f"  Accuracy={acc:.4f}  F1={f1:.4f}  Sens={sens:.4f}  "
              f"Spec={spec:.4f}  AUC={auc:.4f}")
        print(f"  Confusion:\n{cm}")
    return dict(model=name, accuracy=acc, macro_f1=f1, sensitivity=sens,
                specificity=spec, roc_auc=auc, y_true=y_true.tolist(),
                y_pred=y_pred.tolist(),
                y_prob=y_prob.tolist() if y_prob is not None else None)

all_results = []

## 3 · VMD + Sample Entropy Feature Extraction

In [ ]:
def sample_entropy_fast(sig, m=2, r_factor=0.2, max_points=300):
    sig = sig[:max_points]
    N = len(sig)
    r = r_factor * np.std(sig)
    if r == 0:
        return 0.0
    def _phi(m_):
        seqs = np.array([sig[i:i+m_] for i in range(N - m_)])
        count = 0
        for i in range(len(seqs)):
            dist = np.max(np.abs(seqs - seqs[i]), axis=1)
            count += np.sum(dist <= r) - 1
        return count
    A = _phi(m+1); B = _phi(m)
    if B == 0 or A == 0:
        return 0.0
    return float(-np.log(A/B))


def vmd_decompose(sig, K=5, alpha=2000, tau=0, DC=0, init=1, tol=1e-6):
    u, u_hat, omega = VMD(sig, alpha, tau, K, DC, init, tol)
    return u


def extract_vmd_features(epoch, fs=FS, K=5):
    feats = []
    for ch in range(3):
        sig = epoch[ch].astype(np.float64)
        try:
            imfs = vmd_decompose(sig, K=K)
        except Exception:
            feats.extend([0.0] * K)
            continue
        for imf in imfs:
            feats.append(sample_entropy_fast(imf))
    return np.array(feats, dtype=np.float32)


print("⏳ Extracting VMD+SampEn features per subject (concatenated 16s segments)...")
t0 = time.time()

F_vmd_subj = []
for sid in tqdm(unique_sids, desc='VMD per subject', ncols=80):
    mask = ids_all == sid
    subj_epochs = X_all[mask]
    concat_per_ch = np.stack([
        subj_epochs[:, ch, :].flatten() for ch in range(3)
    ])
    concat_trunc = concat_per_ch[:, :4000]
    feats = extract_vmd_features(concat_trunc, K=5)
    F_vmd_subj.append(feats)

F_vmd_subj = np.array(F_vmd_subj, dtype=np.float32)
print(f"\n✅ Done in {(time.time()-t0)/60:.1f} min  shape={F_vmd_subj.shape}")
np.save(os.path.join(OUTPUT_DIR, 'F_vmd_subj.npy'), F_vmd_subj)

## 4 · Reproduce Original Single LOSO Run (Baseline Check)

In [ ]:
logo = LeaveOneGroupOut()
vmd_preds, vmd_probs, vmd_labels = [], [], []

for tr_idx, te_idx in logo.split(F_vmd_subj, y_subj, groups=unique_sids):
    F_tr, F_te = F_vmd_subj[tr_idx], F_vmd_subj[te_idx]
    y_tr, y_te = y_subj[tr_idx], y_subj[te_idx]
    sp = int((y_tr==0).sum()) / max(int((y_tr==1).sum()),1)

    sc = StandardScaler()
    F_tr_sc = sc.fit_transform(F_tr); F_te_sc = sc.transform(F_te)

    lgb_m = lgb.LGBMClassifier(
        n_estimators=150, max_depth=4, learning_rate=0.05,
        scale_pos_weight=sp, random_state=42, verbose=-1)
    lgb_m.fit(F_tr_sc, y_tr)
    p = lgb_m.predict_proba(F_te_sc)[:,1]

    vmd_preds.append(int(p[0]>=0.5))
    vmd_probs.append(float(p[0]))
    vmd_labels.append(int(y_te[0]))

baseline_result = evaluate(
    'VMD+SampEn+LightGBM (original single LOSO, seed=42 default)',
    np.array(vmd_labels), np.array(vmd_preds), np.array(vmd_probs)
)
all_results.append(baseline_result)

## 5 · Seed-Stability Test — LightGBM Internal Randomness

In [ ]:
SEEDS = [42, 7, 123, 2024, 99]
all_seed_results = []

print("Running LOSO-CV with 5 different LightGBM random seeds...\n")

for seed in SEEDS:
    seed_preds, seed_probs, seed_labels = [], [], []

    for tr_idx, te_idx in logo.split(F_vmd_subj, y_subj, groups=unique_sids):
        F_tr, F_te = F_vmd_subj[tr_idx], F_vmd_subj[te_idx]
        y_tr, y_te = y_subj[tr_idx], y_subj[te_idx]
        sp = int((y_tr==0).sum()) / max(int((y_tr==1).sum()),1)

        sc = StandardScaler()
        F_tr_sc = sc.fit_transform(F_tr); F_te_sc = sc.transform(F_te)

        lgb_m = lgb.LGBMClassifier(
            n_estimators=150, max_depth=4, learning_rate=0.05,
            scale_pos_weight=sp, random_state=seed, verbose=-1,
            bagging_fraction=0.8, feature_fraction=0.8, bagging_freq=1)
        lgb_m.fit(F_tr_sc, y_tr)
        p = lgb_m.predict_proba(F_te_sc)[:,1]

        seed_preds.append(int(p[0]>=0.5))
        seed_probs.append(float(p[0]))
        seed_labels.append(int(y_te[0]))

    acc = accuracy_score(seed_labels, seed_preds)
    f1  = f1_score(seed_labels, seed_preds, average='macro', zero_division=0)
    auc = roc_auc_score(seed_labels, seed_probs)
    print(f"  Seed {seed:<5} Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}")
    all_seed_results.append(dict(seed=seed, acc=acc, f1=f1, auc=auc,
                                 probs=seed_probs, labels=seed_labels))

seed_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ['probs','labels']}
                        for r in all_seed_results])
print("\n" + "="*50)
print("Seed-stability summary (LightGBM internal randomness):")
print(seed_df.to_string(index=False, float_format='%.4f'))
print(f"\nMean±Std: Acc={seed_df['acc'].mean():.4f}±{seed_df['acc'].std():.4f}  "
      f"AUC={seed_df['auc'].mean():.4f}±{seed_df['auc'].std():.4f}")

# Average probabilities across seeds for final reportable result
final_probs = np.mean([r['probs'] for r in all_seed_results], axis=0)
final_preds = (final_probs >= 0.5).astype(int)
final_labels = np.array(all_seed_results[0]['labels'])  # same order every time (LOSO is fixed)

all_results.append(evaluate(
    'VMD+SampEn+LightGBM (5-seed averaged, stability-tested)',
    final_labels, final_preds, final_probs
))

## 6 · Additional Check — VMD Feature Extraction Stability

In [ ]:
print("Testing VMD feature stability across different initializations...\n")

vmd_init_results = []

for init_seed in [1, 0]:  # vmdpy supports init=0 (uniform) or init=1 (random)
    np.random.seed(init_seed)
    F_vmd_alt = []
    for sid in tqdm(unique_sids, desc=f'VMD init={init_seed}', ncols=80):
        mask = ids_all == sid
        subj_epochs = X_all[mask]
        concat_per_ch = np.stack([
            subj_epochs[:, ch, :].flatten() for ch in range(3)
        ])
        concat_trunc = concat_per_ch[:, :4000]

        feats = []
        for ch in range(3):
            sig = concat_trunc[ch].astype(np.float64)
            try:
                u, _, _ = VMD(sig, 2000, 0, 5, 0, init_seed, 1e-6)
            except Exception:
                feats.extend([0.0]*5)
                continue
            for imf in u:
                feats.append(sample_entropy_fast(imf))
        F_vmd_alt.append(np.array(feats, dtype=np.float32))

    F_vmd_alt = np.array(F_vmd_alt, dtype=np.float32)

    preds_alt, probs_alt, labels_alt = [], [], []
    for tr_idx, te_idx in logo.split(F_vmd_alt, y_subj, groups=unique_sids):
        F_tr, F_te = F_vmd_alt[tr_idx], F_vmd_alt[te_idx]
        y_tr, y_te = y_subj[tr_idx], y_subj[te_idx]
        sp = int((y_tr==0).sum()) / max(int((y_tr==1).sum()),1)
        sc = StandardScaler()
        F_tr_sc = sc.fit_transform(F_tr); F_te_sc = sc.transform(F_te)
        lgb_m = lgb.LGBMClassifier(n_estimators=150, max_depth=4,
            learning_rate=0.05, scale_pos_weight=sp, random_state=42, verbose=-1)
        lgb_m.fit(F_tr_sc, y_tr)
        p = lgb_m.predict_proba(F_te_sc)[:,1]
        preds_alt.append(int(p[0]>=0.5)); probs_alt.append(float(p[0]))
        labels_alt.append(int(y_te[0]))

    acc_alt = accuracy_score(labels_alt, preds_alt)
    print(f"  VMD init={init_seed}: Acc={acc_alt:.4f}")
    vmd_init_results.append(dict(init=init_seed, acc=acc_alt))

print(f"\nVMD initialization sensitivity: "
      f"{[r['acc'] for r in vmd_init_results]}")
print("If these differ substantially, VMD init is a meaningful noise source.")

## 7 · Final Comparison — Honest Stability-Tested Results

In [ ]:
print("\n" + "="*78)
print("  VMD+SampEn+LightGBM — Stability Analysis Complete")
print("="*78)
print(f"{'Result':<58} {'Acc':>6} {'F1':>6} {'AUC':>6}")
print("─"*78)
for r in all_results:
    name = r['model']
    auc = f"{r['roc_auc']:.3f}" if not np.isnan(r['roc_auc']) else '  N/A'
    print(f"  {name:<56} {r['accuracy']:.3f}  {r['macro_f1']:.3f}  {auc}")

print()
print("  Comparison to other models tested:")
print(f"  CNN-GRU single-run    : 69.1%  →  CNN-GRU 5-seed avg : 65.5% (Δ=-3.6)")
print(f"  VMD+LightGBM single   : {baseline_result['accuracy']*100:.1f}%  "
      f"→  VMD+LightGBM 5-seed avg : {seed_df['acc'].mean()*100:.1f}% "
      f"(Δ={(seed_df['acc'].mean()-baseline_result['accuracy'])*100:+.1f})")
print()
print("  Interpretation:")
delta = seed_df['acc'].mean() - baseline_result['accuracy']
if abs(delta) < 0.02:
    print("  VMD+LightGBM is STABLE — LOSO's fixed fold structure means")
    print("  internal model randomness has minimal effect. This result")
    print("  IS more trustworthy than CNN-GRU's single-run number.")
else:
    print("  VMD+LightGBM also shows meaningful seed sensitivity,")
    print("  consistent with the general small-sample instability we've")
    print("  observed across all models on this n=55 dataset.")
print("="*78)

# Save figure
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(range(len(seed_df)), seed_df['acc'], color='#DD8452', alpha=0.85,
       edgecolor='white')
ax.axhline(seed_df['acc'].mean(), color='#E74C3C', ls='--', lw=2,
           label=f"Mean={seed_df['acc'].mean():.3f}")
ax.axhline(baseline_result['accuracy'], color='#4C72B0', ls=':', lw=2,
           label=f"Original single-run={baseline_result['accuracy']:.3f}")
ax.axhline(0.7225, color='gray', ls=':', lw=1.5, label='Shi et al. 72.25%')
ax.set_xticks(range(len(seed_df)))
ax.set_xticklabels([f'Seed\n{s}' for s in seed_df['seed']])
ax.set_ylabel('Accuracy'); ax.set_ylim(0,1)
ax.set_title('VMD+SampEn+LightGBM — Seed Stability (LightGBM internal randomness)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'vmd_stability.png'), dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Figure saved")